In [0]:
fact_discharges = spark.sql(f"select * from regis_healthcare.silver.discharges;")
fact_discharges.createOrReplaceTempView("discharges")

In [0]:
# Fact_Discharges -- > Source: discharges
# | Foreign Keys       |
# | ------------------ |
# | discharge_key      |
# | resident_key       |
# | facility_key       |
# | discharge_date_key |

from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_discharges = fact_discharges.withColumn("discharge_date_key", date_format(col("discharge_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_discharges = fact_discharges.withColumn("discharge_date_key", col("discharge_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, regexp_replace
fact_discharges = fact_discharges.withColumn(
    "discharge_key",
    regexp_replace(col("discharge_id"), "^DIS", "").cast("int")
)
fact_discharges = fact_discharges.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
fact_discharges = fact_discharges.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# display(df_facilities)
fact_discharges = fact_discharges.select(
 "discharge_key",      
 "resident_key",       
 "facility_key",       
 "discharge_date_key" 
)
display(fact_discharges)


#### cataloge 

In [0]:
fact_discharges.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_discharges")

In [0]:
fact_discharges.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_discharges")
print(fact_discharges.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_discharges")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_discharges")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.discharge_key = source.discharge_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_discharges;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_discharges;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_discharges.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_discharges")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_discharges"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_discharges

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.discharge_key = source.discharge_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
